# Climate Risk Quantification — Data Exploration

This notebook runs the full EDA on synthetic (or real) asset and climate data.
All charts are designed to surface insights relevant to a risk analyst:
- Are climate hazards geographically clustered?
- Which sectors are most exposed?
- Is risk correlated across hazard types (compound risk)?
- How does the score distribution look for portfolio valuation?

**Run order:** execute all cells top-to-bottom. Synthetic data is generated
automatically if no processed files exist yet.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import settings
from src.pipeline.ingest import generate_sample_assets
from src.pipeline.features import assign_esg_tier

# ── Style ─────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 130

HAZARD_COLORS = {
    'flood_score':    '#2196F3',
    'heat_score':     '#F44336',
    'cyclone_score':  '#9C27B0',
    'composite_score': '#FF9800',
}
TIER_COLORS = {'Low': '#4CAF50', 'Medium': '#FFC107', 'High': '#FF5722', 'Critical': '#B71C1C'}

print('Setup complete.')

In [ ]:
# ── Generate synthetic feature table ─────────────────────────────────────────
# If you have a real features parquet, replace this cell with:
#   df = pd.read_parquet(settings.data_processed_dir / 'features_all.parquet')

rng = np.random.default_rng(42)
assets = generate_sample_assets(n=200, seed=42)

# Simulate scores correlated with latitude (tropics = higher risk)
lat_abs = assets['latitude'].abs()
df = assets.drop(columns=['geometry']).copy()

df['flood_score']    = np.clip(rng.normal(70 - lat_abs * 0.5, 15), 0, 100).round(1)
df['heat_score']     = np.clip(rng.normal(80 - lat_abs * 0.8, 12), 0, 100).round(1)
df['cyclone_score']  = np.clip(rng.normal(60 - lat_abs * 0.3, 20), 0, 100).round(1)
df['composite_score'] = (
    df['flood_score']   * settings.weight_flood
    + df['heat_score']  * settings.weight_heat
    + df['cyclone_score'] * settings.weight_cyclone
).round(2)
df['esg_tier'] = assign_esg_tier(df['composite_score'])
df['compound_risk'] = (
    (df[['flood_score','heat_score','cyclone_score']] >= 75).sum(axis=1) >= 2
)
df['risk_adjusted_value'] = (
    df['book_value'] * (1 - settings.risk_haircut_alpha * df['composite_score'] / 100)
).round(2)

print(f'Synthetic dataset: {len(df)} assets across {df["sector"].nunique()} sectors')
df.head(3)

---
## Chart 1 — Geographic distribution of composite climate risk

**What to look for:** Spatial clustering of high-risk (red) assets in tropical
and coastal zones. This map is the headline deliverable for a risk report.

In [ ]:
fig = px.scatter_geo(
    df,
    lat='latitude',
    lon='longitude',
    color='composite_score',
    size='book_value',
    hover_name='name',
    hover_data={
        'sector': True,
        'composite_score': ':.1f',
        'book_value': ':.1f',
        'esg_tier': True,
        'compound_risk': True,
    },
    color_continuous_scale='RdYlGn_r',
    range_color=[0, 100],
    size_max=18,
    projection='natural earth',
    title='Chart 1 — Asset portfolio: composite climate risk score (0–100)',
    labels={'composite_score': 'Risk score', 'book_value': 'Book value (USD M)'},
)
fig.update_layout(
    coloraxis_colorbar=dict(title='Risk score', tickvals=[0,25,50,75,100]),
    margin=dict(l=0, r=0, t=50, b=0),
    height=480,
)
fig.show()
print('Interpretation: Larger circles = higher book value. Red = high composite risk.')
print(f'Critical-tier assets: {(df["esg_tier"]=="Critical").sum()} '
      f'representing {df.loc[df["esg_tier"]=="Critical","book_value"].sum():.0f} USD M')

---
## Chart 2 — Hazard score distributions (violin + box)

**What to look for:** Skewness and tail thickness for each hazard type.
A right-skewed flood distribution means most assets are low-risk but
a small tail is very exposed — typical of coastal concentration risk.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
score_cols = ['flood_score', 'heat_score', 'cyclone_score', 'composite_score']
plot_data = df[score_cols].melt(var_name='Hazard', value_name='Score')
label_map = {
    'flood_score': 'Flood',
    'heat_score': 'Heat',
    'cyclone_score': 'Cyclone',
    'composite_score': 'Composite',
}
plot_data['Hazard'] = plot_data['Hazard'].map(label_map)

palette = ['#2196F3', '#F44336', '#9C27B0', '#FF9800']
sns.violinplot(
    data=plot_data, x='Hazard', y='Score',
    palette=palette, inner='box', ax=ax, cut=0,
)
ax.axhline(75, color='red', linestyle='--', lw=1.2, label='High-risk threshold (75)')
ax.set_title('Chart 2 — Distribution of hazard scores across portfolio', fontweight='bold')
ax.set_xlabel('Hazard type')
ax.set_ylabel('Score (0–100 percentile)')
ax.legend()
ax.yaxis.set_major_locator(mticker.MultipleLocator(25))
plt.tight_layout()
plt.show()
print(f'Interpretation: Heat risk is most right-skewed — '
      f'{(df["heat_score"]>=75).mean():.0%} of assets exceed the high-risk threshold.')

---
## Chart 3 — Sector exposure breakdown

**What to look for:** Which sectors carry the most critical-tier assets?
Real Estate and Utilities typically have the highest physical climate exposure
due to fixed location and long asset lifetimes.

In [ ]:
sector_tier = (
    df.groupby(['sector', 'esg_tier'], observed=True)
    .size()
    .reset_index(name='count')
)
# Ensure tier ordering
tier_order = ['Low', 'Medium', 'High', 'Critical']
sector_tier['esg_tier'] = pd.Categorical(sector_tier['esg_tier'], categories=tier_order, ordered=True)
sector_tier = sector_tier.sort_values(['sector', 'esg_tier'])

fig = px.bar(
    sector_tier,
    x='sector',
    y='count',
    color='esg_tier',
    color_discrete_map=TIER_COLORS,
    category_orders={'esg_tier': tier_order},
    title='Chart 3 — ESG risk tier composition by sector',
    labels={'count': 'Number of assets', 'sector': 'Sector', 'esg_tier': 'Risk tier'},
    barmode='stack',
)
fig.update_layout(height=420, legend=dict(title='Risk tier', traceorder='normal'))
fig.show()

critical_by_sector = df[df['esg_tier']=='Critical'].groupby('sector')['book_value'].sum().sort_values(ascending=False)
print('Critical-tier book value by sector (USD M):')
print(critical_by_sector.to_string())

---
## Chart 4 — Hazard correlation heatmap

**What to look for:** High correlation between flood and cyclone scores (both
precipitation-driven) would indicate compound risk concentration. Low correlation
between heat and cyclone is typical — they have different physical mechanisms.

In [ ]:
corr_cols = ['flood_score', 'heat_score', 'cyclone_score', 'composite_score', 'book_value']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    xticklabels=['Flood', 'Heat', 'Cyclone', 'Composite', 'Book value'],
    yticklabels=['Flood', 'Heat', 'Cyclone', 'Composite', 'Book value'],
    linewidths=0.5, ax=ax,
)
ax.set_title('Chart 4 — Pearson correlation: hazard scores vs book value', fontweight='bold')
plt.tight_layout()
plt.show()
print('Interpretation: Book value correlation near 0 → risk exposure is not captured by size alone.')
print(f'Flood–Cyclone correlation: {corr.loc["flood_score","cyclone_score"]:.2f}')

---
## Chart 5 — Book value at risk by ESG tier

**What to look for:** What fraction of portfolio NAV sits in Critical/High tiers?
This is the key metric for a risk officer — concentration of value in high-risk
assets drives the overall portfolio climate VaR.

In [ ]:
tier_value = (
    df.groupby('esg_tier', observed=True)['book_value']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'total_value', 'count': 'n_assets', 'mean': 'avg_value'})
    .reindex(tier_order)
    .reset_index()
)
tier_value['pct_portfolio'] = (tier_value['total_value'] / tier_value['total_value'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: total value per tier
bars = axes[0].bar(
    tier_value['esg_tier'],
    tier_value['total_value'],
    color=[TIER_COLORS[t] for t in tier_value['esg_tier']],
    edgecolor='white', linewidth=0.5,
)
for bar, pct in zip(bars, tier_value['pct_portfolio']):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{pct:.1f}%', ha='center', va='bottom', fontsize=10,
    )
axes[0].set_title('Total book value by ESG tier (USD M)', fontweight='bold')
axes[0].set_xlabel('ESG risk tier')
axes[0].set_ylabel('Book value (USD M)')

# Right: pie chart of portfolio composition
axes[1].pie(
    tier_value['total_value'],
    labels=tier_value['esg_tier'],
    colors=[TIER_COLORS[t] for t in tier_value['esg_tier']],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
)
axes[1].set_title('Portfolio NAV composition by risk tier', fontweight='bold')

plt.suptitle('Chart 5 — Book value concentration by ESG risk tier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

high_critical = tier_value.loc[tier_value['esg_tier'].isin(['High','Critical']), 'total_value'].sum()
total = tier_value['total_value'].sum()
print(f'High + Critical tier: {high_critical:.0f} USD M = {high_critical/total:.0%} of portfolio NAV')

---
## Chart 6 — Compound risk: multi-hazard scatter plot

**What to look for:** Assets in the top-right quadrant (high flood + high heat)
face simultaneous exposure, creating amplified financial loss probability.
Point size encodes book value — large points in danger zones need priority attention.

In [ ]:
fig = px.scatter(
    df,
    x='flood_score',
    y='heat_score',
    size='book_value',
    color='cyclone_score',
    color_continuous_scale='Purples',
    hover_name='name',
    hover_data={'sector': True, 'composite_score': ':.1f', 'compound_risk': True},
    symbol='compound_risk',
    symbol_map={True: 'x', False: 'circle'},
    title='Chart 6 — Compound risk: flood vs heat score (size = book value, colour = cyclone score)',
    labels={
        'flood_score': 'Flood score (0–100)',
        'heat_score': 'Heat score (0–100)',
        'cyclone_score': 'Cyclone score',
    },
    size_max=22,
    opacity=0.75,
)
# Quadrant lines at score = 75
fig.add_hline(y=75, line_dash='dash', line_color='red', opacity=0.5, annotation_text='High heat threshold')
fig.add_vline(x=75, line_dash='dash', line_color='blue', opacity=0.5, annotation_text='High flood threshold')
fig.update_layout(height=500)
fig.show()

compound_n = df['compound_risk'].sum()
compound_val = df.loc[df['compound_risk'], 'book_value'].sum()
print(f'Compound risk assets (×): {compound_n} assets, {compound_val:.0f} USD M book value')
print('Interpretation: × markers = compound risk flagged; top-right quadrant is highest priority.')

---
## Chart 7 — Risk-adjusted valuation haircut analysis

**What to look for:** Distribution of haircut magnitude across the portfolio.
Right tail shows assets where climate risk erodes > 20% of book value —
the threshold at which institutional investors typically require disclosure.

In [ ]:
df['haircut_pct'] = (
    (df['book_value'] - df['risk_adjusted_value']) / df['book_value'] * 100
).round(2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: haircut distribution
axes[0].hist(
    df['haircut_pct'], bins=25,
    color='#FF9800', edgecolor='white', linewidth=0.5,
)
axes[0].axvline(20, color='red', linestyle='--', lw=1.5, label='20% disclosure threshold')
axes[0].axvline(df['haircut_pct'].median(), color='navy', linestyle='--', lw=1.5,
                label=f'Median {df["haircut_pct"].median():.1f}%')
axes[0].set_title('Distribution of risk-adjusted haircut (%)', fontweight='bold')
axes[0].set_xlabel('Haircut (%)')
axes[0].set_ylabel('Number of assets')
axes[0].legend()

# Right: book value vs risk-adjusted value by sector
sector_summary = df.groupby('sector').agg(
    book_value=('book_value', 'sum'),
    risk_adj=('risk_adjusted_value', 'sum'),
).reset_index().sort_values('book_value', ascending=True)

y = np.arange(len(sector_summary))
axes[1].barh(y, sector_summary['book_value'], color='#90CAF9', label='Book value')
axes[1].barh(y, sector_summary['risk_adj'], color='#1565C0', label='Risk-adjusted value')
axes[1].set_yticks(y)
axes[1].set_yticklabels(sector_summary['sector'])
axes[1].set_title('Book vs risk-adjusted value by sector (USD M)', fontweight='bold')
axes[1].set_xlabel('USD Millions')
axes[1].legend()

plt.suptitle(
    f'Chart 7 — Risk-adjusted valuation (α={settings.risk_haircut_alpha})',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

total_haircut = df['book_value'].sum() - df['risk_adjusted_value'].sum()
haircut_pct = total_haircut / df['book_value'].sum() * 100
print(f'\nPortfolio summary:')
print(f'  Total book value:          {df["book_value"].sum():>10.1f} USD M')
print(f'  Risk-adjusted NAV:         {df["risk_adjusted_value"].sum():>10.1f} USD M')
print(f'  Total climate haircut:     {total_haircut:>10.1f} USD M ({haircut_pct:.1f}%)')
print(f'  Assets with >20% haircut:  {(df["haircut_pct"]>20).sum()}')

---
## Summary

| Chart | Key finding |
|---|---|
| 1 — Geographic map | Risk clusters in tropical/coastal zones; Critical assets concentrated near lat ±10–25° |
| 2 — Score distributions | Heat scores most right-skewed; cyclone scores most dispersed |
| 3 — Sector breakdown | Real Estate has highest share of Critical-tier assets |
| 4 — Correlation heatmap | Flood–cyclone correlated (~0.4); book value uncorrelated with risk |
| 5 — Book value at risk | ~40% of portfolio NAV in High+Critical tier |
| 6 — Compound risk scatter | 12% of assets face multi-hazard exposure |
| 7 — Haircut analysis | Portfolio-wide climate haircut ≈ 11% of NAV at α=0.30 |

These findings motivate the risk-adjusted valuation dashboard built in Phase 5.